# Cenário MRBR | Empresa 4008 + Exercício 2024

| Item | Valor |
|---|---|
| Objeto | `dev_procurement.corp_curated.vw_ds_pro_mrbr_bloq_pagamento_man` (view, DEV) |
| DESCRIBE | `DESCRIBE_MRBR_1.csv`: 32 colunas; tipos STRING, INT e DECIMAL; clustering: não informado |
| Centro | Não existe `cod_centro`; **4008 é empresa, não centro** |
| SAP | Empresa **4008**; Exercício **2024**; outros campos da tela sem preenchimento; opções de Processamento a conferir |
| Base empresa 4008 | 735.219 linhas no HTML exploratório |
| Recorte empresa + exercício | **253.022 linhas** e **250.157 documentos distintos**, medidos na consulta enviada pelo usuário |
| Chave de comparação candidata | `num_doc_faturamento + num_exercicio + cod_empresa`; **não é única no universo**: 20.818 grupos repetidos no HTML |
| View | `DESCRIBE EXTENDED` confirma filtro interno `tp_procedimento_bloqueio = 'Bloq. pagamento man.'` |

### Alertas
| Severidade | Observação |
|---|---|
| 🔴 ALTA | Chave candidata não é única; comparar também multiplicidade por documento, sem deduplicar. |
| 🔴 ALTA | Nomes de fornecedor e usuário são mascarados por hash em amostras e extrações. |
| 🟡 MÉDIA | Datas `dt_lancamento`, `dt_base_prazo_pgto` e `dt_conversao` são STRING `dd/MM/yyyy`. |
| 🟡 MÉDIA | View não possui coluna de centro, grupo de compradores nem vencimento; data-base do prazo não equivale a vencimento. |
| 🟡 MÉDIA | `st_fatura`, `cod_determ_preco` e `ind_nota_existente` são documentados como sem fonte; verificar outras colunas 100% nulas. |
| ⚪ BAIXA | Sem coluna de carga na view: registrar horário da consulta e extrair SAP no mesmo dia. |

**Como usar:** executar Run All no Databricks. Cada SQL é independente e somente leitura. O nome `Centro4008` segue convenção de arquivo, mas **o cenário é por empresa e exercício**. Se o SAP tiver filtro adicional implícito em Processamento, não presumir equivalência antes de confirmar. Não existe filtro por centro neste notebook.

### Índice
0 Recorte; 1 Metadados; 2 Volumetria; 3 Filtros; 4 Granularidade; 5 Duplicidade; 6 Preenchimento; 7 Cardinalidade; 8 Domínio; 9 Indicadores; 10 Numéricos; 11 Totais; 12 Datas; 13 Códigos; 14 Checksum; 15 Amostra; 16 Distribuição; 17 Freshness; 18 Regras; 19 Amostra ampliada; 20 Resumo; 21 Extrações completas.

## 0. Definição do recorte

A empresa substitui o recorte padrão por centro apenas como cenário de negócio, não como equivalência de centro.

In [0]:
SELECT 'MRBR' AS transacao, '4008' AS empresa_sap, '2024' AS exercicio_sap, 'sem coluna de centro' AS limite;

## 1. Metadados

A view já seleciona o procedimento de bloqueio manual; DETAIL e HISTORY não devem ser executados para views.

In [0]:
DESCRIBE EXTENDED `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man`;

## 2. Volumetria

Confere o volume atual e os documentos distintos contra os resultados da consulta enviada.

In [0]:
SELECT COUNT(*) AS linhas_atuais, 253022 AS linhas_esperadas, COUNT(DISTINCT `num_doc_faturamento`) AS documentos_distintos_atuais, 250157 AS documentos_distintos_esperados FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` WHERE `cod_empresa` = '4008' AND `num_exercicio` = '2024';

## 3. Filtro e cobertura

Mostra a grafia literal do exercício e o recorte da empresa; a consulta é limitada à empresa e ao exercício selecionados.

In [0]:
SELECT `cod_empresa`, `num_exercicio`, LENGTH(TRIM(`cod_empresa`)) AS tamanho_empresa, LENGTH(TRIM(`num_exercicio`)) AS tamanho_exercicio, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` WHERE `cod_empresa` = '4008' AND `num_exercicio` = '2024' GROUP BY `cod_empresa`, `num_exercicio`, LENGTH(TRIM(`cod_empresa`)), LENGTH(TRIM(`num_exercicio`));

## 3.1. Funil de filtros

Mede quantas linhas o filtro de exercício retira da base da empresa, sem confundir a base com a extração SAP.

In [0]:
SELECT COUNT(*) AS linhas_empresa_4008, COUNT_IF(`num_exercicio` = '2024') AS linhas_exercicio_2024, COUNT_IF(`num_exercicio` <> '2024' OR `num_exercicio` IS NULL) AS linhas_outros_exercicios FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` WHERE `cod_empresa` = '4008';

## 4. Granularidade

Verifica as chaves documento e documento+exercício+empresa; o exercício é constante no recorte.

In [0]:
WITH a AS (SELECT `num_doc_faturamento` FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` WHERE `cod_empresa` = '4008' AND `num_exercicio` = '2024' GROUP BY `num_doc_faturamento`), b AS (SELECT `num_doc_faturamento`, `num_exercicio`, `cod_empresa` FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` WHERE `cod_empresa` = '4008' AND `num_exercicio` = '2024' GROUP BY `num_doc_faturamento`, `num_exercicio`, `cod_empresa`) SELECT (SELECT COUNT(*) FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` WHERE `cod_empresa` = '4008' AND `num_exercicio` = '2024') AS linhas, (SELECT COUNT(*) FROM a) AS documentos_distintos, (SELECT COUNT(*) FROM b) AS chaves_distintas;

## 5. Duplicatas idênticas e versões

Conta documentos repetidos, cópias idênticas e versões distintas com hash da linha inteira, sem expor valores pessoais.

In [0]:
WITH h AS (SELECT t.`num_doc_faturamento`,t.`num_exercicio`,t.`cod_empresa`,MD5(TO_JSON(STRUCT(t.`num_exercicio`, t.`cod_empresa`, t.`nm_fornecedor`, t.`num_doc_faturamento`, t.`cod_emissor_fatura`, t.`dt_base_prazo_pgto`, t.`dt_lancamento`, t.`qt_dias_desconto_1`, t.`sg_moeda`, t.`sg_moeda_interna`, t.`tp_documento`, t.`cod_bloq_pagamento_log`, t.`nm_usuario`, t.`cod_bloqueio_pgto`, t.`sg_upp`, t.`pct_desconto_2`, t.`pct_desconto_1`, t.`vl_taxa_cambio`, t.`st_fatura`, t.`qt_em_upp`, t.`qt_diferenca`, t.`cod_determ_preco`, t.`cod_pais_destino`, t.`cod_pais_procedencia`, t.`ind_nota_existente`, t.`num_pacote`, t.`ind_bloq_montante_1`, t.`ind_bloq_manual_1`, t.`num_linha_servico`, t.`qt_dias_liquido`, t.`qt_dias_desconto_2`, t.`dt_conversao`), MAP('ignoreNullFields','false'))) AS hash_linha FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` t WHERE `cod_empresa` = '4008' AND `num_exercicio` = '2024'), d AS (SELECT `num_doc_faturamento`,`num_exercicio`,`cod_empresa`,COUNT(*) AS linhas,COUNT(DISTINCT hash_linha) AS versoes FROM h GROUP BY `num_doc_faturamento`,`num_exercicio`,`cod_empresa` HAVING COUNT(*)>1) SELECT COUNT(*) AS chaves_repetidas,COALESCE(SUM(linhas),0) AS linhas_nessas_chaves,COALESCE(SUM(linhas-versoes),0) AS copias_identicas,COUNT_IF(versoes>1) AS chaves_com_versoes_distintas FROM d;

## 5.1. Multiplicidade por documento

Mede a distribuição das repetições sem deduplicar os registros.

In [0]:
WITH d AS (SELECT `num_doc_faturamento`,`num_exercicio`,`cod_empresa`,COUNT(*) AS linhas_por_chave FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` WHERE `cod_empresa` = '4008' AND `num_exercicio` = '2024' GROUP BY `num_doc_faturamento`,`num_exercicio`,`cod_empresa`) SELECT linhas_por_chave,COUNT(*) AS chaves FROM d GROUP BY linhas_por_chave ORDER BY linhas_por_chave;

## 6. Preenchimento de todas as colunas

Uma linha por coluna, com nulos, vazios e zeros; sem publicar nomes.

In [0]:
SELECT p.*,total-nulos-vazios AS preenchidos,CASE WHEN total=0 THEN 'SEM LINHAS' WHEN nulos=total THEN '100% NULO: CONFERIR COMENTARIO' WHEN nulos+vazios=total THEN '100% SEM VALOR: CONFERIR' ELSE 'COM DADOS' END AS veredito FROM (SELECT 'num_exercicio' AS coluna,'string' AS tipo,COUNT(*) AS total,COUNT_IF(`num_exercicio` IS NULL) AS nulos,COUNT_IF(`num_exercicio` IS NOT NULL AND LOWER(TRIM(`num_exercicio`)) IN ('','null','nan','none','#','-')) AS vazios,COUNT_IF(TRIM(`num_exercicio`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` WHERE `cod_empresa` = '4008' AND `num_exercicio` = '2024' UNION ALL SELECT 'cod_empresa' AS coluna,'string' AS tipo,COUNT(*) AS total,COUNT_IF(`cod_empresa` IS NULL) AS nulos,COUNT_IF(`cod_empresa` IS NOT NULL AND LOWER(TRIM(`cod_empresa`)) IN ('','null','nan','none','#','-')) AS vazios,COUNT_IF(TRIM(`cod_empresa`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` WHERE `cod_empresa` = '4008' AND `num_exercicio` = '2024' UNION ALL SELECT 'nm_fornecedor' AS coluna,'string' AS tipo,COUNT(*) AS total,COUNT_IF(`nm_fornecedor` IS NULL) AS nulos,COUNT_IF(`nm_fornecedor` IS NOT NULL AND LOWER(TRIM(`nm_fornecedor`)) IN ('','null','nan','none','#','-')) AS vazios,COUNT_IF(TRIM(`nm_fornecedor`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` WHERE `cod_empresa` = '4008' AND `num_exercicio` = '2024' UNION ALL SELECT 'num_doc_faturamento' AS coluna,'string' AS tipo,COUNT(*) AS total,COUNT_IF(`num_doc_faturamento` IS NULL) AS nulos,COUNT_IF(`num_doc_faturamento` IS NOT NULL AND LOWER(TRIM(`num_doc_faturamento`)) IN ('','null','nan','none','#','-')) AS vazios,COUNT_IF(TRIM(`num_doc_faturamento`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` WHERE `cod_empresa` = '4008' AND `num_exercicio` = '2024' UNION ALL SELECT 'cod_emissor_fatura' AS coluna,'string' AS tipo,COUNT(*) AS total,COUNT_IF(`cod_emissor_fatura` IS NULL) AS nulos,COUNT_IF(`cod_emissor_fatura` IS NOT NULL AND LOWER(TRIM(`cod_emissor_fatura`)) IN ('','null','nan','none','#','-')) AS vazios,COUNT_IF(TRIM(`cod_emissor_fatura`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` WHERE `cod_empresa` = '4008' AND `num_exercicio` = '2024' UNION ALL SELECT 'dt_base_prazo_pgto' AS coluna,'string' AS tipo,COUNT(*) AS total,COUNT_IF(`dt_base_prazo_pgto` IS NULL) AS nulos,COUNT_IF(`dt_base_prazo_pgto` IS NOT NULL AND LOWER(TRIM(`dt_base_prazo_pgto`)) IN ('','null','nan','none','#','-')) AS vazios,COUNT_IF(TRIM(`dt_base_prazo_pgto`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` WHERE `cod_empresa` = '4008' AND `num_exercicio` = '2024' UNION ALL SELECT 'dt_lancamento' AS coluna,'string' AS tipo,COUNT(*) AS total,COUNT_IF(`dt_lancamento` IS NULL) AS nulos,COUNT_IF(`dt_lancamento` IS NOT NULL AND LOWER(TRIM(`dt_lancamento`)) IN ('','null','nan','none','#','-')) AS vazios,COUNT_IF(TRIM(`dt_lancamento`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` WHERE `cod_empresa` = '4008' AND `num_exercicio` = '2024' UNION ALL SELECT 'qt_dias_desconto_1' AS coluna,'int' AS tipo,COUNT(*) AS total,COUNT_IF(`qt_dias_desconto_1` IS NULL) AS nulos,CAST(0 AS BIGINT) AS vazios,COUNT_IF(`qt_dias_desconto_1`=0) AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` WHERE `cod_empresa` = '4008' AND `num_exercicio` = '2024' UNION ALL SELECT 'sg_moeda' AS coluna,'string' AS tipo,COUNT(*) AS total,COUNT_IF(`sg_moeda` IS NULL) AS nulos,COUNT_IF(`sg_moeda` IS NOT NULL AND LOWER(TRIM(`sg_moeda`)) IN ('','null','nan','none','#','-')) AS vazios,COUNT_IF(TRIM(`sg_moeda`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` WHERE `cod_empresa` = '4008' AND `num_exercicio` = '2024' UNION ALL SELECT 'sg_moeda_interna' AS coluna,'string' AS tipo,COUNT(*) AS total,COUNT_IF(`sg_moeda_interna` IS NULL) AS nulos,COUNT_IF(`sg_moeda_interna` IS NOT NULL AND LOWER(TRIM(`sg_moeda_interna`)) IN ('','null','nan','none','#','-')) AS vazios,COUNT_IF(TRIM(`sg_moeda_interna`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` WHERE `cod_empresa` = '4008' AND `num_exercicio` = '2024' UNION ALL SELECT 'tp_documento' AS coluna,'string' AS tipo,COUNT(*) AS total,COUNT_IF(`tp_documento` IS NULL) AS nulos,COUNT_IF(`tp_documento` IS NOT NULL AND LOWER(TRIM(`tp_documento`)) IN ('','null','nan','none','#','-')) AS vazios,COUNT_IF(TRIM(`tp_documento`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` WHERE `cod_empresa` = '4008' AND `num_exercicio` = '2024' UNION ALL SELECT 'cod_bloq_pagamento_log' AS coluna,'string' AS tipo,COUNT(*) AS total,COUNT_IF(`cod_bloq_pagamento_log` IS NULL) AS nulos,COUNT_IF(`cod_bloq_pagamento_log` IS NOT NULL AND LOWER(TRIM(`cod_bloq_pagamento_log`)) IN ('','null','nan','none','#','-')) AS vazios,COUNT_IF(TRIM(`cod_bloq_pagamento_log`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` WHERE `cod_empresa` = '4008' AND `num_exercicio` = '2024' UNION ALL SELECT 'nm_usuario' AS coluna,'string' AS tipo,COUNT(*) AS total,COUNT_IF(`nm_usuario` IS NULL) AS nulos,COUNT_IF(`nm_usuario` IS NOT NULL AND LOWER(TRIM(`nm_usuario`)) IN ('','null','nan','none','#','-')) AS vazios,COUNT_IF(TRIM(`nm_usuario`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` WHERE `cod_empresa` = '4008' AND `num_exercicio` = '2024' UNION ALL SELECT 'cod_bloqueio_pgto' AS coluna,'string' AS tipo,COUNT(*) AS total,COUNT_IF(`cod_bloqueio_pgto` IS NULL) AS nulos,COUNT_IF(`cod_bloqueio_pgto` IS NOT NULL AND LOWER(TRIM(`cod_bloqueio_pgto`)) IN ('','null','nan','none','#','-')) AS vazios,COUNT_IF(TRIM(`cod_bloqueio_pgto`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` WHERE `cod_empresa` = '4008' AND `num_exercicio` = '2024' UNION ALL SELECT 'sg_upp' AS coluna,'string' AS tipo,COUNT(*) AS total,COUNT_IF(`sg_upp` IS NULL) AS nulos,COUNT_IF(`sg_upp` IS NOT NULL AND LOWER(TRIM(`sg_upp`)) IN ('','null','nan','none','#','-')) AS vazios,COUNT_IF(TRIM(`sg_upp`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` WHERE `cod_empresa` = '4008' AND `num_exercicio` = '2024' UNION ALL SELECT 'pct_desconto_2' AS coluna,'string' AS tipo,COUNT(*) AS total,COUNT_IF(`pct_desconto_2` IS NULL) AS nulos,COUNT_IF(`pct_desconto_2` IS NOT NULL AND LOWER(TRIM(`pct_desconto_2`)) IN ('','null','nan','none','#','-')) AS vazios,COUNT_IF(TRIM(`pct_desconto_2`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` WHERE `cod_empresa` = '4008' AND `num_exercicio` = '2024' UNION ALL SELECT 'pct_desconto_1' AS coluna,'string' AS tipo,COUNT(*) AS total,COUNT_IF(`pct_desconto_1` IS NULL) AS nulos,COUNT_IF(`pct_desconto_1` IS NOT NULL AND LOWER(TRIM(`pct_desconto_1`)) IN ('','null','nan','none','#','-')) AS vazios,COUNT_IF(TRIM(`pct_desconto_1`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` WHERE `cod_empresa` = '4008' AND `num_exercicio` = '2024' UNION ALL SELECT 'vl_taxa_cambio' AS coluna,'string' AS tipo,COUNT(*) AS total,COUNT_IF(`vl_taxa_cambio` IS NULL) AS nulos,COUNT_IF(`vl_taxa_cambio` IS NOT NULL AND LOWER(TRIM(`vl_taxa_cambio`)) IN ('','null','nan','none','#','-')) AS vazios,COUNT_IF(TRIM(`vl_taxa_cambio`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` WHERE `cod_empresa` = '4008' AND `num_exercicio` = '2024' UNION ALL SELECT 'st_fatura' AS coluna,'string' AS tipo,COUNT(*) AS total,COUNT_IF(`st_fatura` IS NULL) AS nulos,COUNT_IF(`st_fatura` IS NOT NULL AND LOWER(TRIM(`st_fatura`)) IN ('','null','nan','none','#','-')) AS vazios,COUNT_IF(TRIM(`st_fatura`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` WHERE `cod_empresa` = '4008' AND `num_exercicio` = '2024' UNION ALL SELECT 'qt_em_upp' AS coluna,'decimal(18,3)' AS tipo,COUNT(*) AS total,COUNT_IF(`qt_em_upp` IS NULL) AS nulos,CAST(0 AS BIGINT) AS vazios,COUNT_IF(`qt_em_upp`=0) AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` WHERE `cod_empresa` = '4008' AND `num_exercicio` = '2024' UNION ALL SELECT 'qt_diferenca' AS coluna,'decimal(18,3)' AS tipo,COUNT(*) AS total,COUNT_IF(`qt_diferenca` IS NULL) AS nulos,CAST(0 AS BIGINT) AS vazios,COUNT_IF(`qt_diferenca`=0) AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` WHERE `cod_empresa` = '4008' AND `num_exercicio` = '2024' UNION ALL SELECT 'cod_determ_preco' AS coluna,'string' AS tipo,COUNT(*) AS total,COUNT_IF(`cod_determ_preco` IS NULL) AS nulos,COUNT_IF(`cod_determ_preco` IS NOT NULL AND LOWER(TRIM(`cod_determ_preco`)) IN ('','null','nan','none','#','-')) AS vazios,COUNT_IF(TRIM(`cod_determ_preco`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` WHERE `cod_empresa` = '4008' AND `num_exercicio` = '2024' UNION ALL SELECT 'cod_pais_destino' AS coluna,'string' AS tipo,COUNT(*) AS total,COUNT_IF(`cod_pais_destino` IS NULL) AS nulos,COUNT_IF(`cod_pais_destino` IS NOT NULL AND LOWER(TRIM(`cod_pais_destino`)) IN ('','null','nan','none','#','-')) AS vazios,COUNT_IF(TRIM(`cod_pais_destino`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` WHERE `cod_empresa` = '4008' AND `num_exercicio` = '2024' UNION ALL SELECT 'cod_pais_procedencia' AS coluna,'string' AS tipo,COUNT(*) AS total,COUNT_IF(`cod_pais_procedencia` IS NULL) AS nulos,COUNT_IF(`cod_pais_procedencia` IS NOT NULL AND LOWER(TRIM(`cod_pais_procedencia`)) IN ('','null','nan','none','#','-')) AS vazios,COUNT_IF(TRIM(`cod_pais_procedencia`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` WHERE `cod_empresa` = '4008' AND `num_exercicio` = '2024' UNION ALL SELECT 'ind_nota_existente' AS coluna,'string' AS tipo,COUNT(*) AS total,COUNT_IF(`ind_nota_existente` IS NULL) AS nulos,COUNT_IF(`ind_nota_existente` IS NOT NULL AND LOWER(TRIM(`ind_nota_existente`)) IN ('','null','nan','none','#','-')) AS vazios,COUNT_IF(TRIM(`ind_nota_existente`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` WHERE `cod_empresa` = '4008' AND `num_exercicio` = '2024' UNION ALL SELECT 'num_pacote' AS coluna,'string' AS tipo,COUNT(*) AS total,COUNT_IF(`num_pacote` IS NULL) AS nulos,COUNT_IF(`num_pacote` IS NOT NULL AND LOWER(TRIM(`num_pacote`)) IN ('','null','nan','none','#','-')) AS vazios,COUNT_IF(TRIM(`num_pacote`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` WHERE `cod_empresa` = '4008' AND `num_exercicio` = '2024' UNION ALL SELECT 'ind_bloq_montante_1' AS coluna,'string' AS tipo,COUNT(*) AS total,COUNT_IF(`ind_bloq_montante_1` IS NULL) AS nulos,COUNT_IF(`ind_bloq_montante_1` IS NOT NULL AND LOWER(TRIM(`ind_bloq_montante_1`)) IN ('','null','nan','none','#','-')) AS vazios,COUNT_IF(TRIM(`ind_bloq_montante_1`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` WHERE `cod_empresa` = '4008' AND `num_exercicio` = '2024' UNION ALL SELECT 'ind_bloq_manual_1' AS coluna,'string' AS tipo,COUNT(*) AS total,COUNT_IF(`ind_bloq_manual_1` IS NULL) AS nulos,COUNT_IF(`ind_bloq_manual_1` IS NOT NULL AND LOWER(TRIM(`ind_bloq_manual_1`)) IN ('','null','nan','none','#','-')) AS vazios,COUNT_IF(TRIM(`ind_bloq_manual_1`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` WHERE `cod_empresa` = '4008' AND `num_exercicio` = '2024' UNION ALL SELECT 'num_linha_servico' AS coluna,'string' AS tipo,COUNT(*) AS total,COUNT_IF(`num_linha_servico` IS NULL) AS nulos,COUNT_IF(`num_linha_servico` IS NOT NULL AND LOWER(TRIM(`num_linha_servico`)) IN ('','null','nan','none','#','-')) AS vazios,COUNT_IF(TRIM(`num_linha_servico`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` WHERE `cod_empresa` = '4008' AND `num_exercicio` = '2024' UNION ALL SELECT 'qt_dias_liquido' AS coluna,'int' AS tipo,COUNT(*) AS total,COUNT_IF(`qt_dias_liquido` IS NULL) AS nulos,CAST(0 AS BIGINT) AS vazios,COUNT_IF(`qt_dias_liquido`=0) AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` WHERE `cod_empresa` = '4008' AND `num_exercicio` = '2024' UNION ALL SELECT 'qt_dias_desconto_2' AS coluna,'int' AS tipo,COUNT(*) AS total,COUNT_IF(`qt_dias_desconto_2` IS NULL) AS nulos,CAST(0 AS BIGINT) AS vazios,COUNT_IF(`qt_dias_desconto_2`=0) AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` WHERE `cod_empresa` = '4008' AND `num_exercicio` = '2024' UNION ALL SELECT 'dt_conversao' AS coluna,'string' AS tipo,COUNT(*) AS total,COUNT_IF(`dt_conversao` IS NULL) AS nulos,COUNT_IF(`dt_conversao` IS NOT NULL AND LOWER(TRIM(`dt_conversao`)) IN ('','null','nan','none','#','-')) AS vazios,COUNT_IF(TRIM(`dt_conversao`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` WHERE `cod_empresa` = '4008' AND `num_exercicio` = '2024') p ORDER BY coluna;

## 6.1. Lacunas documentadas

Campos com #GAP-SemFonte ou indisponibilidade de CDS são medidos sem rotular automaticamente como erro.

In [0]:
SELECT * FROM (SELECT 'st_fatura' AS coluna,COUNT(*) AS total,COUNT_IF(`st_fatura` IS NULL) AS nulos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` WHERE `cod_empresa` = '4008' AND `num_exercicio` = '2024' UNION ALL SELECT 'qt_diferenca' AS coluna,COUNT(*) AS total,COUNT_IF(`qt_diferenca` IS NULL) AS nulos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` WHERE `cod_empresa` = '4008' AND `num_exercicio` = '2024' UNION ALL SELECT 'cod_determ_preco' AS coluna,COUNT(*) AS total,COUNT_IF(`cod_determ_preco` IS NULL) AS nulos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` WHERE `cod_empresa` = '4008' AND `num_exercicio` = '2024' UNION ALL SELECT 'ind_nota_existente' AS coluna,COUNT(*) AS total,COUNT_IF(`ind_nota_existente` IS NULL) AS nulos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` WHERE `cod_empresa` = '4008' AND `num_exercicio` = '2024' UNION ALL SELECT 'num_pacote' AS coluna,COUNT(*) AS total,COUNT_IF(`num_pacote` IS NULL) AS nulos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` WHERE `cod_empresa` = '4008' AND `num_exercicio` = '2024' UNION ALL SELECT 'num_linha_servico' AS coluna,COUNT(*) AS total,COUNT_IF(`num_linha_servico` IS NULL) AS nulos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` WHERE `cod_empresa` = '4008' AND `num_exercicio` = '2024') x ORDER BY coluna;

## 7. Cardinalidade

Compara documentos, emissores e códigos de bloqueio sem expor nomes.

In [0]:
SELECT COUNT(DISTINCT `num_doc_faturamento`) AS documentos,COUNT(DISTINCT `cod_emissor_fatura`) AS emissores,COUNT(DISTINCT `cod_bloq_pagamento_log`) AS bloqueios_logisticos,COUNT(DISTINCT `cod_bloqueio_pgto`) AS bloqueios_fi FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` WHERE `cod_empresa` = '4008' AND `num_exercicio` = '2024';

## 8. Domínio

Top oito valores de categorias de negócio, sem códigos de pessoa.

In [0]:
WITH d AS (SELECT 'tp_documento' AS coluna,CAST(`tp_documento` AS STRING) AS valor,COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` WHERE `cod_empresa` = '4008' AND `num_exercicio` = '2024' GROUP BY `tp_documento` UNION ALL SELECT 'sg_moeda' AS coluna,CAST(`sg_moeda` AS STRING) AS valor,COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` WHERE `cod_empresa` = '4008' AND `num_exercicio` = '2024' GROUP BY `sg_moeda` UNION ALL SELECT 'cod_bloq_pagamento_log' AS coluna,CAST(`cod_bloq_pagamento_log` AS STRING) AS valor,COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` WHERE `cod_empresa` = '4008' AND `num_exercicio` = '2024' GROUP BY `cod_bloq_pagamento_log` UNION ALL SELECT 'cod_bloqueio_pgto' AS coluna,CAST(`cod_bloqueio_pgto` AS STRING) AS valor,COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` WHERE `cod_empresa` = '4008' AND `num_exercicio` = '2024' GROUP BY `cod_bloqueio_pgto`) SELECT coluna,valor,linhas FROM (SELECT d.*,ROW_NUMBER() OVER(PARTITION BY coluna ORDER BY linhas DESC,valor) AS pos FROM d) x WHERE pos<=8 ORDER BY coluna,pos;

## 9. Indicadores de bloqueio

Distingue códigos logísticos e financeiros; não presumir que seus significados sejam idênticos.

In [0]:
SELECT `cod_bloq_pagamento_log`,`cod_bloqueio_pgto`,COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` WHERE `cod_empresa` = '4008' AND `num_exercicio` = '2024' GROUP BY `cod_bloq_pagamento_log`,`cod_bloqueio_pgto` ORDER BY linhas DESC LIMIT 100;

## 10. Perfil numérico

Perfila quantidades numéricas sem interpretar campos STRING como valores.

In [0]:
SELECT * FROM (SELECT 'qt_dias_desconto_1' AS coluna,CAST(MIN(`qt_dias_desconto_1`) AS DOUBLE) AS minimo,CAST(MAX(`qt_dias_desconto_1`) AS DOUBLE) AS maximo,CAST(AVG(`qt_dias_desconto_1`) AS DOUBLE) AS media,CAST(PERCENTILE_APPROX(`qt_dias_desconto_1`,0.5) AS DOUBLE) AS mediana,COUNT_IF(`qt_dias_desconto_1`<0) AS negativos,COUNT_IF(`qt_dias_desconto_1`=0) AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` WHERE `cod_empresa` = '4008' AND `num_exercicio` = '2024' UNION ALL SELECT 'qt_em_upp' AS coluna,CAST(MIN(`qt_em_upp`) AS DOUBLE) AS minimo,CAST(MAX(`qt_em_upp`) AS DOUBLE) AS maximo,CAST(AVG(`qt_em_upp`) AS DOUBLE) AS media,CAST(PERCENTILE_APPROX(`qt_em_upp`,0.5) AS DOUBLE) AS mediana,COUNT_IF(`qt_em_upp`<0) AS negativos,COUNT_IF(`qt_em_upp`=0) AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` WHERE `cod_empresa` = '4008' AND `num_exercicio` = '2024' UNION ALL SELECT 'qt_diferenca' AS coluna,CAST(MIN(`qt_diferenca`) AS DOUBLE) AS minimo,CAST(MAX(`qt_diferenca`) AS DOUBLE) AS maximo,CAST(AVG(`qt_diferenca`) AS DOUBLE) AS media,CAST(PERCENTILE_APPROX(`qt_diferenca`,0.5) AS DOUBLE) AS mediana,COUNT_IF(`qt_diferenca`<0) AS negativos,COUNT_IF(`qt_diferenca`=0) AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` WHERE `cod_empresa` = '4008' AND `num_exercicio` = '2024' UNION ALL SELECT 'qt_dias_liquido' AS coluna,CAST(MIN(`qt_dias_liquido`) AS DOUBLE) AS minimo,CAST(MAX(`qt_dias_liquido`) AS DOUBLE) AS maximo,CAST(AVG(`qt_dias_liquido`) AS DOUBLE) AS media,CAST(PERCENTILE_APPROX(`qt_dias_liquido`,0.5) AS DOUBLE) AS mediana,COUNT_IF(`qt_dias_liquido`<0) AS negativos,COUNT_IF(`qt_dias_liquido`=0) AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` WHERE `cod_empresa` = '4008' AND `num_exercicio` = '2024' UNION ALL SELECT 'qt_dias_desconto_2' AS coluna,CAST(MIN(`qt_dias_desconto_2`) AS DOUBLE) AS minimo,CAST(MAX(`qt_dias_desconto_2`) AS DOUBLE) AS maximo,CAST(AVG(`qt_dias_desconto_2`) AS DOUBLE) AS media,CAST(PERCENTILE_APPROX(`qt_dias_desconto_2`,0.5) AS DOUBLE) AS mediana,COUNT_IF(`qt_dias_desconto_2`<0) AS negativos,COUNT_IF(`qt_dias_desconto_2`=0) AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` WHERE `cod_empresa` = '4008' AND `num_exercicio` = '2024') x ORDER BY coluna;

## 11. Totais para conciliação

Soma cada quantidade isoladamente; conferir a granularidade antes de comparar totais.

In [0]:
SELECT * FROM (SELECT 'qt_dias_desconto_1' AS coluna,CAST(SUM(`qt_dias_desconto_1`) AS DOUBLE) AS total FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` WHERE `cod_empresa` = '4008' AND `num_exercicio` = '2024' UNION ALL SELECT 'qt_em_upp' AS coluna,CAST(SUM(`qt_em_upp`) AS DOUBLE) AS total FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` WHERE `cod_empresa` = '4008' AND `num_exercicio` = '2024' UNION ALL SELECT 'qt_diferenca' AS coluna,CAST(SUM(`qt_diferenca`) AS DOUBLE) AS total FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` WHERE `cod_empresa` = '4008' AND `num_exercicio` = '2024' UNION ALL SELECT 'qt_dias_liquido' AS coluna,CAST(SUM(`qt_dias_liquido`) AS DOUBLE) AS total FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` WHERE `cod_empresa` = '4008' AND `num_exercicio` = '2024' UNION ALL SELECT 'qt_dias_desconto_2' AS coluna,CAST(SUM(`qt_dias_desconto_2`) AS DOUBLE) AS total FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` WHERE `cod_empresa` = '4008' AND `num_exercicio` = '2024') x ORDER BY coluna;

## 12. Datas STRING

Converte apenas dd/MM/yyyy; mede sentinelas, formatos inválidos e janela cronológica sem comparar literais.

In [0]:
SELECT * FROM (SELECT 'dt_lancamento' AS coluna,MIN(TRY_TO_DATE(`dt_lancamento`,'dd/MM/yyyy')) AS primeira_data,MAX(TRY_TO_DATE(`dt_lancamento`,'dd/MM/yyyy')) AS ultima_data,COUNT_IF(`dt_lancamento` IS NOT NULL AND TRY_TO_DATE(`dt_lancamento`,'dd/MM/yyyy') IS NULL) AS invalidas,COUNT_IF(TRIM(`dt_lancamento`)='00000000') AS sentinela_00000000 FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` WHERE `cod_empresa` = '4008' AND `num_exercicio` = '2024' UNION ALL SELECT 'dt_base_prazo_pgto' AS coluna,MIN(TRY_TO_DATE(`dt_base_prazo_pgto`,'dd/MM/yyyy')) AS primeira_data,MAX(TRY_TO_DATE(`dt_base_prazo_pgto`,'dd/MM/yyyy')) AS ultima_data,COUNT_IF(`dt_base_prazo_pgto` IS NOT NULL AND TRY_TO_DATE(`dt_base_prazo_pgto`,'dd/MM/yyyy') IS NULL) AS invalidas,COUNT_IF(TRIM(`dt_base_prazo_pgto`)='00000000') AS sentinela_00000000 FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` WHERE `cod_empresa` = '4008' AND `num_exercicio` = '2024' UNION ALL SELECT 'dt_conversao' AS coluna,MIN(TRY_TO_DATE(`dt_conversao`,'dd/MM/yyyy')) AS primeira_data,MAX(TRY_TO_DATE(`dt_conversao`,'dd/MM/yyyy')) AS ultima_data,COUNT_IF(`dt_conversao` IS NOT NULL AND TRY_TO_DATE(`dt_conversao`,'dd/MM/yyyy') IS NULL) AS invalidas,COUNT_IF(TRIM(`dt_conversao`)='00000000') AS sentinela_00000000 FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` WHERE `cod_empresa` = '4008' AND `num_exercicio` = '2024') x ORDER BY coluna;

## 13. Códigos e zeros à esquerda

Identifica possíveis colisões ao normalizar documento e emissor, sem alterar os dados originais.

In [0]:
SELECT * FROM (SELECT 'num_doc_faturamento' AS coluna,COUNT_IF(TRIM(`num_doc_faturamento`) RLIKE '^0+[0-9]+$') AS com_zeros,COUNT(DISTINCT CASE WHEN TRIM(`num_doc_faturamento`) RLIKE '^[0-9]+$' THEN TRIM(`num_doc_faturamento`) END) AS codigos_numericos,COUNT(DISTINCT CASE WHEN TRIM(`num_doc_faturamento`) RLIKE '^[0-9]+$' THEN REGEXP_REPLACE(TRIM(`num_doc_faturamento`),'^0+(?=[0-9])','') END) AS codigos_normalizados FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` WHERE `cod_empresa` = '4008' AND `num_exercicio` = '2024' UNION ALL SELECT 'cod_emissor_fatura' AS coluna,COUNT_IF(TRIM(`cod_emissor_fatura`) RLIKE '^0+[0-9]+$') AS com_zeros,COUNT(DISTINCT CASE WHEN TRIM(`cod_emissor_fatura`) RLIKE '^[0-9]+$' THEN TRIM(`cod_emissor_fatura`) END) AS codigos_numericos,COUNT(DISTINCT CASE WHEN TRIM(`cod_emissor_fatura`) RLIKE '^[0-9]+$' THEN REGEXP_REPLACE(TRIM(`cod_emissor_fatura`),'^0+(?=[0-9])','') END) AS codigos_normalizados FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` WHERE `cod_empresa` = '4008' AND `num_exercicio` = '2024') x ORDER BY coluna;

## 14. Checksum

Conta cópias 100% idênticas sem exibir hashes.

In [0]:
WITH h AS (SELECT MD5(TO_JSON(STRUCT(t.`num_exercicio`, t.`cod_empresa`, t.`nm_fornecedor`, t.`num_doc_faturamento`, t.`cod_emissor_fatura`, t.`dt_base_prazo_pgto`, t.`dt_lancamento`, t.`qt_dias_desconto_1`, t.`sg_moeda`, t.`sg_moeda_interna`, t.`tp_documento`, t.`cod_bloq_pagamento_log`, t.`nm_usuario`, t.`cod_bloqueio_pgto`, t.`sg_upp`, t.`pct_desconto_2`, t.`pct_desconto_1`, t.`vl_taxa_cambio`, t.`st_fatura`, t.`qt_em_upp`, t.`qt_diferenca`, t.`cod_determ_preco`, t.`cod_pais_destino`, t.`cod_pais_procedencia`, t.`ind_nota_existente`, t.`num_pacote`, t.`ind_bloq_montante_1`, t.`ind_bloq_manual_1`, t.`num_linha_servico`, t.`qt_dias_liquido`, t.`qt_dias_desconto_2`, t.`dt_conversao`), MAP('ignoreNullFields','false'))) AS hash_linha FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` t WHERE `cod_empresa` = '4008' AND `num_exercicio` = '2024'), d AS (SELECT hash_linha,COUNT(*) AS linhas FROM h GROUP BY hash_linha) SELECT COALESCE(SUM(linhas),0) AS linhas,COUNT(*) AS linhas_distintas,COALESCE(SUM(linhas-1),0) AS copias_identicas FROM d;

## 15. Amostra

20 linhas sem nomes literais.

In [0]:
SELECT `num_exercicio`, `cod_empresa`, `num_doc_faturamento`, `cod_emissor_fatura`, `dt_base_prazo_pgto`, `dt_lancamento`, `qt_dias_desconto_1`, `sg_moeda`, `sg_moeda_interna`, `tp_documento`, `cod_bloq_pagamento_log`, `cod_bloqueio_pgto`, `sg_upp`, `pct_desconto_2`, `pct_desconto_1`, `vl_taxa_cambio`, `st_fatura`, `qt_em_upp`, `qt_diferenca`, `cod_determ_preco`, `cod_pais_destino`, `cod_pais_procedencia`, `ind_nota_existente`, `num_pacote`, `ind_bloq_montante_1`, `ind_bloq_manual_1`, `num_linha_servico`, `qt_dias_liquido`, `qt_dias_desconto_2`, `dt_conversao`, SUBSTRING(MD5(CAST(`nm_fornecedor` AS STRING)),1,8) AS `nm_fornecedor_hash`, SUBSTRING(MD5(CAST(`nm_usuario` AS STRING)),1,8) AS `nm_usuario_hash` FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` WHERE `cod_empresa` = '4008' AND `num_exercicio` = '2024' ORDER BY `num_doc_faturamento` LIMIT 20;

## 16. Distribuição interna

Combina tipo de documento, moeda e bloqueio sem agregar usuários.

In [0]:
SELECT `tp_documento`,`sg_moeda`,`cod_bloq_pagamento_log`,COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` WHERE `cod_empresa` = '4008' AND `num_exercicio` = '2024' GROUP BY `tp_documento`,`sg_moeda`,`cod_bloq_pagamento_log` ORDER BY linhas DESC LIMIT 100;

## 17. Freshness

Sem coluna de ingestão nesta view; registrar instante da consulta e volume.

In [0]:
SELECT CURRENT_TIMESTAMP() AS instante_consulta,COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` WHERE `cod_empresa` = '4008' AND `num_exercicio` = '2024';

## 18. Regras específicas MRBR

Compara códigos de bloqueio, data de lançamento e campos sem fonte sem inferir causa.

In [0]:
SELECT COUNT(*) AS linhas,COUNT_IF(`cod_bloq_pagamento_log` IS NULL OR TRIM(`cod_bloq_pagamento_log`)='') AS sem_bloqueio_log,COUNT_IF(`cod_bloqueio_pgto` IS NULL OR TRIM(`cod_bloqueio_pgto`)='') AS sem_bloqueio_fi,COUNT_IF(`st_fatura` IS NULL) AS status_sem_fonte,COUNT_IF(`dt_lancamento` IS NOT NULL AND TRY_TO_DATE(`dt_lancamento`,'dd/MM/yyyy') IS NULL) AS lancamentos_invalidos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` WHERE `cod_empresa` = '4008' AND `num_exercicio` = '2024';

## 19. Amostra ampliada

500 linhas, sem nomes literais.

In [0]:
SELECT `num_exercicio`, `cod_empresa`, `num_doc_faturamento`, `cod_emissor_fatura`, `dt_base_prazo_pgto`, `dt_lancamento`, `qt_dias_desconto_1`, `sg_moeda`, `sg_moeda_interna`, `tp_documento`, `cod_bloq_pagamento_log`, `cod_bloqueio_pgto`, `sg_upp`, `pct_desconto_2`, `pct_desconto_1`, `vl_taxa_cambio`, `st_fatura`, `qt_em_upp`, `qt_diferenca`, `cod_determ_preco`, `cod_pais_destino`, `cod_pais_procedencia`, `ind_nota_existente`, `num_pacote`, `ind_bloq_montante_1`, `ind_bloq_manual_1`, `num_linha_servico`, `qt_dias_liquido`, `qt_dias_desconto_2`, `dt_conversao`, SUBSTRING(MD5(CAST(`nm_fornecedor` AS STRING)),1,8) AS `nm_fornecedor_hash`, SUBSTRING(MD5(CAST(`nm_usuario` AS STRING)),1,8) AS `nm_usuario_hash` FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` WHERE `cod_empresa` = '4008' AND `num_exercicio` = '2024' ORDER BY `num_doc_faturamento` LIMIT 500;

## 20. Resumo do cenário

Bloco de conferência com volume, documentos e repetição da chave; 253022 é medição enviada, não garantia futura.

In [0]:
WITH d AS (SELECT `num_doc_faturamento`,`num_exercicio`,`cod_empresa`,COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` WHERE `cod_empresa` = '4008' AND `num_exercicio` = '2024' GROUP BY `num_doc_faturamento`,`num_exercicio`,`cod_empresa`) SELECT 'MRBR' AS transacao,'4008' AS empresa,'2024' AS exercicio,253022 AS volume_esperado,(SELECT COUNT(*) FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` WHERE `cod_empresa` = '4008' AND `num_exercicio` = '2024') AS volume_atual,(SELECT COUNT(*) FROM d) AS chaves_distintas,(SELECT COUNT(*) FROM d WHERE linhas>1) AS chaves_repetidas;

## 21. EXTRAÇÃO COMPLETA: base empresa 4008

Base de diagnóstico para o funil. Não é equivalente ao filtro Exercício 2024 da tela SAP; não exportar fora de ambiente autorizado.

In [0]:
SELECT CASE WHEN TRIM(`num_doc_faturamento`) RLIKE '^[0-9]+$' THEN REGEXP_REPLACE(TRIM(`num_doc_faturamento`),'^0+(?=[0-9])','') ELSE TRIM(`num_doc_faturamento`) END AS `num_doc_faturamento_normalizado`,`num_exercicio`, `cod_empresa`, `num_doc_faturamento`, `cod_emissor_fatura`, `dt_base_prazo_pgto`, `dt_lancamento`, `qt_dias_desconto_1`, `sg_moeda`, `sg_moeda_interna`, `tp_documento`, `cod_bloq_pagamento_log`, `cod_bloqueio_pgto`, `sg_upp`, `pct_desconto_2`, `pct_desconto_1`, `vl_taxa_cambio`, `st_fatura`, `qt_em_upp`, `qt_diferenca`, `cod_determ_preco`, `cod_pais_destino`, `cod_pais_procedencia`, `ind_nota_existente`, `num_pacote`, `ind_bloq_montante_1`, `ind_bloq_manual_1`, `num_linha_servico`, `qt_dias_liquido`, `qt_dias_desconto_2`, `dt_conversao`, SUBSTRING(MD5(CAST(`nm_fornecedor` AS STRING)),1,8) AS `nm_fornecedor_hash`, SUBSTRING(MD5(CAST(`nm_usuario` AS STRING)),1,8) AS `nm_usuario_hash` FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` WHERE `cod_empresa` = '4008' ORDER BY `num_exercicio`,`num_doc_faturamento`;

## 21.1. EXTRAÇÃO COMPLETA: equivalente aos filtros explícitos SAP

Empresa 4008 + Exercício 2024. Se a seção Processamento do SAP acrescentar filtros, esta equivalência permanece pendente.

In [0]:
SELECT CASE WHEN TRIM(`num_doc_faturamento`) RLIKE '^[0-9]+$' THEN REGEXP_REPLACE(TRIM(`num_doc_faturamento`),'^0+(?=[0-9])','') ELSE TRIM(`num_doc_faturamento`) END AS `num_doc_faturamento_normalizado`,`num_exercicio`, `cod_empresa`, `num_doc_faturamento`, `cod_emissor_fatura`, `dt_base_prazo_pgto`, `dt_lancamento`, `qt_dias_desconto_1`, `sg_moeda`, `sg_moeda_interna`, `tp_documento`, `cod_bloq_pagamento_log`, `cod_bloqueio_pgto`, `sg_upp`, `pct_desconto_2`, `pct_desconto_1`, `vl_taxa_cambio`, `st_fatura`, `qt_em_upp`, `qt_diferenca`, `cod_determ_preco`, `cod_pais_destino`, `cod_pais_procedencia`, `ind_nota_existente`, `num_pacote`, `ind_bloq_montante_1`, `ind_bloq_manual_1`, `num_linha_servico`, `qt_dias_liquido`, `qt_dias_desconto_2`, `dt_conversao`, SUBSTRING(MD5(CAST(`nm_fornecedor` AS STRING)),1,8) AS `nm_fornecedor_hash`, SUBSTRING(MD5(CAST(`nm_usuario` AS STRING)),1,8) AS `nm_usuario_hash` FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` WHERE `cod_empresa` = '4008' AND `num_exercicio` = '2024' ORDER BY `num_doc_faturamento`,`num_exercicio`,`cod_empresa`;

## Checklist de comparação SAP

- [ ] Na tela MRBR, selecionar Empresa **4008** e Exercício **2024**; deixar outros filtros de seleção vazios.
- [ ] Conferir opções da seção **Processamento** e filtros implícitos antes de afirmar equivalência com a view.
- [ ] Registrar instante da execução; exportar SAP e Datalake no mesmo dia.
- [ ] Comparar **253.022 linhas** e **250.157 documentos distintos** com os valores atuais, sem eliminar documentos repetidos.
- [ ] Conferir chave, duplicatas puras versus versões, zeros à esquerda, datas `dd/MM/yyyy` e campos sem fonte.
- [ ] Não divulgar nomes de fornecedor ou usuário literais; comparar hashes somente se o tratamento de normalização for igual dos dois lados.
- [ ] `Centro4008` no nome é convenção legada; **4008 é empresa, não centro**.